### Zoteroize and Obsidianize a Perplexity Dialogue

In a Perplexity dialogue copied to the clipboard by the perplexity copy button and then saved to a file, replace 
the citation numbers with matching Obsidian literature note or Zotero item links

In [1]:
import re
import pathlib as pl
import sys
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz

%load_ext autoreload
%autoreload 2

In [2]:
def split_body_source(perplexity_file: pl.Path):
    """Replace links in standard Perplexity (saved clipboard) output with links 
    to Zotero items or Obsidian lit notes."""    
    
    content = perplexity_file.read_text(encoding='utf-8')
    section_parts = content.split("\nCitations:\n", 1)
    if len(section_parts) < 2:
        print("Missing citations")
        body, citations = section_parts, ""
    else:
        body, citations = section_parts
    
    source_matches = list(re.finditer(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', citations, flags=re.M))

    url_to_source_num = {m.group('url'): m.group('num') for m in source_matches}
    
    return body, url_to_source_num, source_matches

def relink_chunks(body, url_to_source_num, source_matches, relinked_file: pl.Path) -> None:

    def make_relinks_from_source(cite_num: str, doc_url: str) -> str:
        """Returns what a relinked citation would look like if present in the body,
        given a source part citation number and url.  Also appends to the global list, 
        relinked_sources, a relinked source part link.  Expects the global set, body_cite_nums."""
        
        numbered_link = f"[{cite_num}]({doc_url})"
        if zotero_item := relinker.find_zotero_item_via_url(doc_url):
            body_link = relinker.create_obsidian_or_zotero_link(zotero_item)
            relinked_sources.append(f'({numbered_link}) **{body_link}**')
        else:
            body_link = f"=={numbered_link}==" # mark it as "not in zotero"
            source_line = f'({numbered_link}) {doc_url}'
            source_line = f'=={source_line} ==' if cite_num in body_cite_nums else source_line
            relinked_sources.append(source_line)
            
        return body_link

    relinker = lpz.ZoteroLinkConverter()
    relinked_sources = []
    
    body_cite_nums = set(re.findall(r'\[(\d+)\]', body))
    # source_num_to_link = {url: make_relinks_from_source(num, url) for url, num in url_to_source_num.items()}
    
    ic(list(source_matches))

    source_num_to_link = {m.group('num'): make_relinks_from_source(m.group('num'), m.group('url'))
                          for m in source_matches }
    
    body_relinked = re.sub(r'\[(\d+)\]', 
                           lambda m: f' {source_num_to_link.get(m.group(1))}', body)
    sources_relinked = "\n".join(relinked_sources)
    
    relinked_file.write_text(f"{body_relinked}\nCitations:\n{sources_relinked}", 
                             encoding='utf-8')

def relink_perplexity_export(perplexity_file: pl.Path, relinked_file: pl.Path) -> None:
    #source_matches = [] # a global
    body, url_to_source_num, source_matches = split_body_source(perplexity_file)
    relink_chunks(body, url_to_source_num, source_matches, relinked_file)    

In [3]:
perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
output_file = rfw.refwrangle_test_dir / 'tmp' / "tmp_new_cites_perplexity_example.md"
print(f'{perplexity_dialog_file=}\n-->\n{output_file=}')

relink_perplexity_export(perplexity_dialog_file, output_file)
print('Done.')

perplexity_dialog_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/perplexity_example.md')
-->
output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_new_cites_perplexity_example.md')
Reading from cache.
Done.


In [ ]:
from collections import defaultdict
import pandas as pd

# get the sources from all docs to be merged
all_bodies, all_url_to_source_nums = [], []
for doc in [perplexity_dialog_file]:
    body, url_to_source_num = split_body_source(doc)
    all_bodies.append(body)
    all_url_to_source_nums.append(url_to_source_num)

# find all the doc citations for each unique URL
allurls = defaultdict(list)
print(allurls)
for docIx, url_to_source_num in enumerate(all_url_to_source_nums):
    for url, num in url_to_source_num.items():
        allurls[url].append(dict(orig_num=num, docIx=docIx))

# create new citenums for a combined document with a combined sources section
new_cite_num = 1
lut = []
for url, infos in allurls.items():
    for info in infos:
        lut.append({'url': url, 'new_cite_num': str(new_cite_num)} | info)
    new_cite_num += 1

lut = pd.DataFrame(lut).set_index(['docIx', 'orig_num'])
all_new_cite_nums = lut.new_cite_num.unique()


# replace the cite numbers in the combined cite numbers or links to my stuff
#for docIx, body in enumerate(all_bodies):
    
    #new_body_cite_nums = {old_num: new:lut[old_num] for old_num in body_cite_nums}
    
    #def renumber_cites(body)
    #source_num_to_link = {url: make_relinks_from_source(num, url) for url, num in url_to_source_num.items()}


#lut

defaultdict(<class 'list'>, {})


In [ ]:
#all_new_cite_nums

In [17]:
display(lut)
#lut.loc[0, '3'].new_cite_num
all_new_cite_nums

url new_cite_num
docIx orig_num                                                                
0     1         https://www.quantilope.com/resources/what-is-b...            1
      2         https://www.meltwater.com/en/blog/reputation-m...            2
      3                  https://brand24.com/blog/brand-tracking/            3
      4         https://zorgle.co.uk/how-a-strong-brand-image-...            4
      5         https://www.hanoverresearch.com/insights-blog/...            5
      6         https://www.groupcaliber.com/brand-tracking-10...            6
      7         https://pmc.ncbi.nlm.nih.gov/articles/PMC5095155/            7
      8         https://vorecol.com/blogs/blog-the-impact-of-r...            8
      9         https://pure.manchester.ac.uk/ws/portalfiles/p...            9
      10        https://pmc.ncbi.nlm.nih.gov/articles/PMC9167012/           10
      11                 https://www.mdpi.com/2076-3417/9/16/3336           11
      12        https://www.quantilope.com/resources/guide-to-...           12
      13        https://pure.roehampton.ac.uk/ws/files/1360152...           13
      14        https://www.ijitee.org/wp-content/uploads/pape...           14
      15        https://www.researchgate.net/publication/23371...           15
      16          https://prlab.co/blog/csr-and-public-relations/           16
      17        https://www.linkedin.com/advice/3/what-most-ef...           17
      18        https://survicate.com/blog/brand-sentiment-ana...           18
      19        https://www.channelfutures.com/regulation-comp...           19
      20        https://www.driveresearch.com/market-research-...           20
      21                https://brand24.com/blog/brand-sentiment/           21
      22        https://www.mackenziecorp.com/4-brand-developm...           22
      23        https://www.3epr.com/public-relations-kpis-you...           23
      24        https://www.pewresearch.org/internet/2021/02/1...           24
      25         https://www.mightyroar.com/blog/promotion-profit           25
      26        https://ijecm.co.uk/wp-content/uploads/2016/03...           26
      27        https://www.inputkit.io/en/blog/bad-brand-imag...           27
      28        https://www.forbes.com/councils/forbescommunic...           28
      29        https://kellercenter.hankamer.baylor.edu/news/...           29
      30        https://www.brandtastic1.com/blog/companys-ima...           30
      31        https://agicap.com/en/article/profit-and-loss-...           31
      32        http://www.diva-portal.org/smash/get/diva2:133...           32
      33        https://www.linkedin.com/advice/3/how-can-you-...           33
      34        https://www.meltwater.com/en/blog/top-social-m...           34
      35             https://www.surveymonkey.com/mp/brand-image/           35
      36        https://www.sprinklr.com/blog/brand-monitoring...           36
      37                https://www.eskimi.com/blog/brand-metrics           37
      38        https://www.qualtrics.com/experience-managemen...           38
      39        https://www.pemavor.com/top-brand-monitoring-t...           39
      40               https://www.ronsela.com/brand-measurement/           40
      41        https://blog.tapresearch.com/high-frequency-br...           41
      42        https://youscan.io/blog/social-media-monitorin...           42
      43                  https://www.bestviso.com/en/brandimage/           43
      44        https://www.aimtechnologies.co/pr-monitoring-t...           44
      45        https://www.aimtechnologies.co/changing-brand-...           45
      46        https://www.forbes.com/councils/forbescommunic...           46
      47        https://dto-research.com/en/current/profession...           47
      48        https://www.forbes.com/sites/forbesagencycounc...           48
      49        https://www.agilitypr.com/pr-news/public-relat...           49
      50        https://hbr.org/2

array(['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12',
       '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23',
       '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34',
       '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45',
       '46', '47', '48', '49', '50', '51', '52', '53', '54'], dtype=object)

In [7]:
import re

# Input string and regex pattern
text = "apple banana apple orange banana"
pattern = r'\b(\w+)\b'  # Matches words

# Step 1: Extract all matches
matches = re.findall(pattern, text)

# Step 2: Compute unique substitutes
unique_substitutes = {match: f"word_{i}" for i, match in enumerate(set(matches), start=1)}

# Step 3: Define replacement function
def replacement_function(match):
    return unique_substitutes[match.group(0)]

# Step 4: Perform substitutions
result = re.sub(pattern, replacement_function, text)

print("Original:", text)
print("Modified:", result)


Original: apple banana apple orange banana
Modified: word_1 word_2 word_1 word_3 word_2


In [8]:
# def relink_perplexity_export(perplexity_file: pl.Path, relinked_file: pl.Path) -> None:
#     """Replace links in standard Perplexity (saved clipboard) output with links 
#     to Zotero items or Obsidian lit notes."""    
    
#     relinker = lpz.ZoteroLinkConverter()

#     def make_relinks_from_source(cite_num: str, doc_url: str) -> str:
#         """Returns what a relinked citation would look like if present in the body,
#         given a source part citation number and url.  Also appends to the global list, 
#         relinked_sources, a relinked source part link.  Expects the global set, body_cite_nums."""
        
#         numbered_link = f"[{cite_num}]({doc_url})"
#         if zotero_item := relinker.find_zotero_item_via_url(doc_url):
#             body_link = relinker.create_obsidian_or_zotero_link(zotero_item)
#             relinked_sources.append(f'({numbered_link}) **{body_link}**')
#         else:
#             body_link = f"=={numbered_link}==" # mark it as "not in zotero"
#             source_line = f'({numbered_link}) {doc_url}'
#             source_line = f'=={source_line} ==' if cite_num in body_cite_nums else source_line
#             relinked_sources.append(source_line)
            
#         return body_link
    
#     content = perplexity_file.read_text(encoding='utf-8')
#     section_parts = content.split("\nCitations:\n", 1)
#     if len(section_parts) < 2:
#         print("Missing citations")
#         body, citations = section_parts, ""
#     else:
#         body, citations = section_parts
    
#     source_matches = re.finditer(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', citations, flags=re.M)

#     relinked_sources = []
#     body_cite_nums = set(re.findall(r'\[(\d+)\]', body))
#     url_to_source_num = {m.group('url'): m.group('num') for m in source_matches}
#     #ic(len(url_to_source_num))
#     source_num_to_link = {url: make_relinks_from_source(num, url) for url, num in url_to_source_num.items()}
#     #ic(len(source_num_to_link), len(url_to_source_num))
#     # source_num_to_link = {m.group('num'): make_relinks_from_source(m.group('num'), m.group('url'))
#     #                       for m in source_matches }
    
#     body_relinked = re.sub(r'\[(\d+)\]', 
#                            lambda m: f' {source_num_to_link.get(m.group(1))}', body)
#     sources_relinked = "\n".join(relinked_sources)
    
#     relinked_text = f"{body_relinked}\nCitations:\n{sources_relinked}"
#     relinked_file.write_text(f"{body_relinked}\nCitations:\n{sources_relinked}", 
#                              encoding='utf-8')
    
#     return relinked_text, source_num_to_link